# Baseline ML 

Este notebook replica el flujo simple del reel (cargar datos → features → split temporal → entrenar → evaluar):
- Datos desde `DataCleaner`/`preprocess_data`.
- Target **direction_2h** (0=DOWN, 1=FLAT, 2=UP).
- Split temporal + `TimeSeriesSplit` para selección de features.

> **Nota seguridad:** no hardcodear claves API en el repo. Usa variables de entorno.


In [1]:
import numpy as np
import pandas as pd
import sys
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

sys.path.append(os.path.abspath("../src"))
from data_cleaner import DataCleanerConfig, DataCleaner, preprocess_data


In [2]:
cfg_5m = DataCleanerConfig(
    source="alpaca",
    symbol=["QQQ", "TLT", "VXX", "BNDX"],
    interval="5m",
    start_date="2022-01-01",
    end_date="2026-01-02",
)

cleaner = DataCleaner(cfg_5m)
df_raw = cleaner.cargar_datos()
df = preprocess_data(df_raw, "qqq")

df.head()


c:\Users\koki2\Desktop\TRADING_ALG\src\data_cleaner.py:242: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df["return"] = df["close"].pct_change()


,datetime,open_bndx,open_qqq,open_tlt,open_vxx,high_bndx,high_qqq,high_tlt,high_vxx,low_bndx,...,log_return_lag5,vol_rolling,rsi,macd,macd_signal,ema_9,ema_21,ema_50,ema_100,ema_200
0,2022-01-03 14:25:00+00:00,55.06,399.38,146.55,18.2100,55.060,399.41,146.600,18.30,55.06,...,-0.000275,0.000512,21.777778,-0.000707,-0.000518,399.110000,399.110000,399.110000,399.110000,399.110000
1,2022-01-03 14:30:00+00:00,55.03,399.05,146.41,18.2700,55.060,400.68,146.500,18.29,55.01,...,0.000601,0.000921,50.991501,-0.000474,-0.000509,399.372000,399.229091,399.161373,399.135941,399.123035
2,2022-01-03 14:35:00+00:00,55.02,400.44,146.36,18.1200,55.035,401.53,146.595,18.16,55.01,...,-0.000025,0.000983,60.246913,-0.000158,-0.000439,399.709600,399.395537,399.235829,399.174041,399.142308
3,2022-01-03 14:40:00+00:00,55.03,401.07,146.57,18.0701,55.040,401.26,146.800,18.15,55.03,...,0.000326,0.001024,57.547170,-0.000008,-0.000353,399.879680,399.501397,399.287757,399.201486,399.156414
4,2022-01-03 14:45:00+00:00,55.03,400.57,146.53,18.1100,55.050,400.75,146.940,18.56,55.03,...,-0.000050,0.001669,36.489076,-0.000359,-0.000354,399.547244,399.384679,399.245786,399.182001,399.147072


In [17]:
HORIZON = 24  # 2h en velas de 5m
eps = 1e-12

# features
df["return"] = df["close"].pct_change(fill_method=None).fillna(0.0)
df["logret"] = np.log(df["close"].clip(lower=eps)).diff()

df["volatility"] = df["return"].rolling(5).std()
df["vol_5"]  = df["logret"].rolling(5).std()
df["vol_20"] = df["logret"].rolling(20).std()

for w in [5, 10, 20, 50, 100, 200]:
    df[f"ma_{w}"] = df["close"].rolling(w).mean()
    df[f"dist_ma_{w}"] = df["close"] / (df[f"ma_{w}"] + eps) - 1

if "rsi" not in df.columns:
    n = 14
    delta = df["close"].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    avg_loss = loss.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    df["rsi"] = 100 - (100 / (1 + rs))

for sym in ["tlt", "vxx", "bndx"]:
    c = f"close_{sym}"
    r = f"ret_{sym}"
    if c in df.columns:
        df[r] = df[c].pct_change()

for lag in [1, 3, 6, 12]:
    df[f"return_lag{lag}"] = df["return"].shift(lag)
    df[f"rsi_lag{lag}"]    = df["rsi"].shift(lag)

# --- target ---
df["close_future"] = df["close"].shift(-HORIZON)
df["direction_2h"] = (df["close_future"] > df["close"]).astype(int)

df = df.dropna().reset_index(drop=True)


In [18]:
features = [
    "close","volume","return","logret","vol_5","vol_20",
    "ma_5","ma_10","ma_20","ma_50","ma_100","ma_200",
    "dist_ma_5","dist_ma_10","dist_ma_20","dist_ma_50","dist_ma_100","dist_ma_200",
    "ret_tlt","ret_vxx","ret_bndx",
    "rsi","return_lag1","return_lag3","return_lag6","return_lag12",
    "rsi_lag1","rsi_lag3","rsi_lag6","rsi_lag12"
]

# qué features te faltan (por si algún exógeno no está)
missing = [c for c in features if c not in df.columns]
print("missing:", missing)

features_ok = [c for c in features if c in df.columns]

X = df[features_ok]
y = df["direction_2h"]

y.value_counts()


missing: []


direction_2h
1    42898
0    36546
Name: count, dtype: int64

In [26]:
df["return"] = df["close"].pct_change(fill_method=None).fillna(0.0)
X = df.select_dtypes(include=[np.number]).drop(columns=["close_future","direction_2h"], errors="ignore")
y = df["direction_2h"].astype(int)

selected_lstm = [
    "close", "volume",
    "return", "logret", "vol_20",
    "rsi", "return_lag3", "rsi_lag6",
    "dist_ma_20", "dist_ma_50", "dist_ma_100", "dist_ma_200",
    "ret_tlt","ret_vxx","ret_bndx"
]

Xsel = X[selected_lstm]


In [27]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)


In [28]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

minmax_features = ["close", "volume", "rsi", "rsi_lag6"]
standard_features = [
    "return", "logret", "vol_20", "return_lag3",
    "dist_ma_20", "dist_ma_50", "dist_ma_100", "dist_ma_200",
    "ret_tlt", "ret_vxx", "ret_bndx"
]


scalers = {}
for col in minmax_features:
    scalers[col] = MinMaxScaler(feature_range=(0, 1))  

for col in standard_features:
    scalers[col] = StandardScaler()

# Split + Scaler

In [29]:
import numpy as np

# Ventana de pasado que mira la LSTM
LOOKBACK = 96

# Split por tiempo: 80% train, 20% test
split = int(len(Xsel) * 0.8)

# 1) Copiamos las features
X_scaled_df = Xsel.copy()

# 2) Escalamos columna por columna (fit SOLO con train)
for col in Xsel.columns:
    sc = scalers[col]                      # tu scaler ya definido (MinMax o Standard)
    sc.fit(Xsel[[col]].iloc[:split])       # fit solo con train (importante)
    X_scaled_df[col] = sc.transform(Xsel[[col]])[:, 0]  # transform a toda la serie

# 3) A numpy para Keras
X_scaled = X_scaled_df.to_numpy(dtype=np.float32)

# y en numpy (0/1)
y_arr = y.to_numpy().astype(np.float32)


# Secuencias para la LSTM

In [34]:
Xs, ys, idxs = [], [], []

# Recorremos el tiempo y armamos ventanas de tamaño LOOKBACK
for i in range(LOOKBACK, len(X_scaled)):
    Xs.append(X_scaled[i-LOOKBACK+1:i+1])  # ventana de features
    ys.append(y_arr[i])                # etiqueta del "momento i"
    idxs.append(i)                     # índice real (para split correcto)

Xs = np.array(Xs, dtype=np.float32)
ys = np.array(ys, dtype=np.float32)
idxs = np.array(idxs)

# Split por tiempo usando el índice original (sin mezclar)
train_mask = idxs < split
X_train, y_train = Xs[train_mask], ys[train_mask]
X_test,  y_test  = Xs[~train_mask], ys[~train_mask]

print(X_train.shape, X_test.shape)


(63459, 96, 15) (15889, 96, 15)


In [35]:
!pip install tensorflow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [38]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

# Val = último 10% del train (sin shuffle)
cut = int(len(X_train) * 0.9)
X_tr, y_tr = X_train[:cut], y_train[:cut]
X_val, y_val = X_train[cut:], y_train[cut:]

# Modelo MINIMALISTA
model = Sequential([
    LSTM(32, input_shape=(LOOKBACK, X_train.shape[2])),
    Dense(1, activation="sigmoid")  # porque y es 0/1 (direction_2h)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1.0),
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

es = EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=64,
    shuffle=False,          # time series: no mezclar
    callbacks=[es],
    verbose=1
)

print(model.evaluate(X_test, y_test, verbose=0))


Epoch 1/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 13s 14ms/step - accuracy: 0.5403 - auc: 0.5301 - loss: 0.6926 - val_accuracy: 0.5176 - val_auc: 0.4815 - val_loss: 0.6958
Epoch 2/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.5454 - auc: 0.5383 - loss: 0.6888 - val_accuracy: 0.5186 - val_auc: 0.4779 - val_loss: 0.6955
Epoch 3/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.5465 - auc: 0.5456 - loss: 0.6875 - val_accuracy: 0.5091 - val_auc: 0.4787 - val_loss: 0.6958
Epoch 4/20
893/893 ━━━━━━━━━━━━━━━━━━━━ 12s 13ms/step - accuracy: 0.5467 - auc: 0.5497 - loss: 0.6867 - val_accuracy: 0.5102 - val_auc: 0.4809 - val_loss: 0.6960
[0.6868412494659424, 0.5528982281684875, 0.5098978281021118]


In [39]:
p = y_test.mean()
print("Positivos:", p, "Baseline acc:", max(p, 1-p))


Positivos: 0.56321985 Baseline acc: 0.56321985
